In [ ]:
!pip -q install pillow-heif exifread pandas tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import random
import math
from pathlib import Path
from math import radians, cos, sin, asin, sqrt

import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import exifread
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()

In [ ]:
# move kyle photo db to photo_database_heic
source_folder = "/content/drive/Shareddrives/CIS5190_final_project/kyle_photo_db"
target_folder = "/content/drive/Shareddrives/CIS5190_final_project/photo_database_heic"
moved = 0
skipped = 0

for name in os.listdir(source_folder):
    src_path = os.path.join(source_folder, name)
    dst_path = os.path.join(target_folder, name)

    if os.path.isfile(src_path):
        if os.path.exists(dst_path):
            print(f"Skip (same name exists): {name}")
            skipped += 1
        else:
            shutil.move(src_path, dst_path)
            moved += 1


Done. Moved 1143 files, skipped 0 files.


In [ ]:
BASE_DIR = "/content/drive/Shareddrives/CIS5190_final_project"

# raw HEIC photo dataset dir
RAW_DIR = os.path.join(BASE_DIR, "photo_database_heic")

# raw jpg photo dataset dir
JPG_DIR = os.path.join(BASE_DIR, "photo_database_jpg")

# final splitted dataset dir
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_dataset")
TRAIN_DIR = os.path.join(PROCESSED_DIR, "train")
VAL_DIR = os.path.join(PROCESSED_DIR, "validation")
TEST_DIR = os.path.join(PROCESSED_DIR, "test")

# metadata
ALL_METADATA_CSV = os.path.join(BASE_DIR, "all_metadata.csv")
CONVERSION_LOG_CSV = os.path.join(BASE_DIR, "conversion_log.csv")
MISSING_GPS_CSV = os.path.join(BASE_DIR, "missing_gps.csv")

In [ ]:
# make the folders
for folder in [JPG_DIR, PROCESSED_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
# incase of duplicated name:
def get_unique_output_path(folder, filename):
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(folder, filename)
    counter = 1
    while os.path.exists(candidate):
        candidate = os.path.join(folder, f"{base}_{counter}{ext}")
        counter += 1
    return candidate

In [ ]:
# Get all files in HEIC
all_source_files = []
for root, dirs, files in os.walk(RAW_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1]
        if ext == ".HEIC":
            all_source_files.append(os.path.join(root, fname))

print("Total image files found:", len(all_source_files))

Total image files found: 3292


In [ ]:
# convert heic to jpg
conversion_log = []

for src_path in tqdm(all_source_files):
    src_name = os.path.basename(src_path)
    stem, ext = os.path.splitext(src_name)

    try:
        out_name = f"{stem}.jpg"
        dst_path = get_unique_output_path(JPG_DIR, out_name)

        if ext == ".HEIC":
            img = Image.open(src_path)
            exif_bytes = img.info.get("exif", None)

            if img.mode != "RGB":
                img = img.convert("RGB")

            if exif_bytes:
                img.save(dst_path, format="JPEG", quality=95, exif=exif_bytes)
            else:
                img.save(dst_path, format="JPEG", quality=95)

            conversion_log.append((src_name, os.path.basename(dst_path), "converted"))

    except Exception as e:
        conversion_log.append((src_name, None, f"failed: {e}"))


100%|██████████| 3292/3292 [1:49:16<00:00,  1.99s/it]


In [ ]:
BASE_DIR = "/content/drive/Shareddrives/CIS5190_final_project"

IMAGE_DIR = os.path.join(BASE_DIR, "photo_database_jpg")

METADATA_PATH = os.path.join(BASE_DIR, "photo_database_jpg/metadata.csv")

GROUPED_DATASET_DIR = os.path.join(BASE_DIR, "processed_dataset_grouped")
TRAIN_DIR = os.path.join(GROUPED_DATASET_DIR, "train")
VAL_DIR = os.path.join(GROUPED_DATASET_DIR, "validation")
TEST_DIR = os.path.join(GROUPED_DATASET_DIR, "test")

In [ ]:
for folder in [GROUPED_DATASET_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Folders ready.")

Folders ready.


In [ ]:
df = pd.read_csv(METADATA_PATH)
print(df.head())
print("Total images:", len(df))

      file_name   Latitude  Longitude
0  IMG_3787.jpg  39.950508 -75.191903
1  IMG_3782.jpg  39.950536 -75.191942
2  IMG_3785.jpg  39.950517 -75.191894
3  IMG_3788.jpg  39.950508 -75.191881
4  IMG_3789.jpg  39.950506 -75.191881
Total images: 3292


In [ ]:
def haversine_meters(lat1, lon1, lat2, lon2):
    R = 6371000  # aarth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1

In [ ]:
coords = df[["Latitude", "Longitude"]].values
n = len(df)
uf = UnionFind(n)

threshold_m = 1.5

for i in tqdm(range(n)):
    lat1, lon1 = coords[i]
    for j in range(i + 1, n):
        lat2, lon2 = coords[j]
        dist = haversine_meters(lat1, lon1, lat2, lon2)
        if dist <= threshold_m:
            uf.union(i, j)

print("Grouping finished.")

100%|██████████| 3292/3292 [00:19<00:00, 169.51it/s]

Grouping finished.


In [ ]:
group_ids = [uf.find(i) for i in range(n)]
df["group_id_raw"] = group_ids

unique_groups = {gid: idx for idx, gid in enumerate(sorted(df["group_id_raw"].unique()))}
df["group_id"] = df["group_id_raw"].map(unique_groups)

df = df.drop(columns=["group_id_raw"])

print(df.head())
print("Number of groups:", df["group_id"].nunique())

      file_name   Latitude  Longitude  group_id
0  IMG_3787.jpg  39.950508 -75.191903         0
1  IMG_3782.jpg  39.950536 -75.191942         1
2  IMG_3785.jpg  39.950517 -75.191894         0
3  IMG_3788.jpg  39.950508 -75.191881         2
4  IMG_3789.jpg  39.950506 -75.191881         2
Number of groups: 663


In [ ]:
import random
import pandas as pd

SEED = 42
random.seed(SEED)

# num of pics per group
group_sizes = df.groupby("group_id").size().reset_index(name="count")

# total pics & proportion
total_images = group_sizes["count"].sum()
target_train = 0.70 * total_images
target_val = 0.15 * total_images
target_test = 0.15 * total_images

# rand + sort with size
group_sizes = group_sizes.sample(frac=1, random_state=SEED).sort_values(
    by="count", ascending=False
).reset_index(drop=True)

train_groups, val_groups, test_groups = [], [], []
train_count, val_count, test_count = 0, 0, 0

for _, row in group_sizes.iterrows():
    gid = row["group_id"]
    c = row["count"]

    # how many pic left for each split
    deficits = {
        "train": target_train - train_count,
        "val": target_val - val_count,
        "test": target_test - test_count,
    }

    # send the group to the split lacks pic
    chosen_split = max(deficits, key=deficits.get)

    if chosen_split == "train":
        train_groups.append(gid)
        train_count += c
    elif chosen_split == "val":
        val_groups.append(gid)
        val_count += c
    else:
        test_groups.append(gid)
        test_count += c

print("Train groups:", len(train_groups))
print("Validation groups:", len(val_groups))
print("Test groups:", len(test_groups))

print("Train images:", train_count, f"({train_count / total_images:.2%})")
print("Validation images:", val_count, f"({val_count / total_images:.2%})")
print("Test images:", test_count, f"({test_count / total_images:.2%})")
print("Total images:", train_count + val_count + test_count)

Train groups: 291
Validation groups: 186
Test groups: 186
Train images: 2304 (69.99%)
Validation images: 494 (15.01%)
Test images: 494 (15.01%)
Total images: 3292


Since our split is performed at the group level rather than the individual-image level, a naive random split over group IDs does not guarantee that the final number of images in train, validation, and test will follow the intended 70/15/15 ratio. To address this, we first compute the size of each location group, shuffle groups with a fixed random seed, sort them by size, and then greedily assign each group to the split that is currently furthest below its target image count. This preserves group integrity while producing a split that better matches the desired image-level proportions.

In [ ]:
train_df = df[df["group_id"].isin(train_groups)].copy()
val_df = df[df["group_id"].isin(val_groups)].copy()
test_df = df[df["group_id"].isin(test_groups)].copy()

In [ ]:
def copy_split_images(split_df, src_folder, dst_folder):
    copied = 0
    missing = 0

    for fname in tqdm(split_df["file_name"].tolist()):
        src = os.path.join(src_folder, fname)
        dst = os.path.join(dst_folder, fname)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing += 1
            print("Missing:", fname)

    return copied, missing

In [ ]:
train_copied, train_missing = copy_split_images(train_df, IMAGE_DIR, TRAIN_DIR)
val_copied, val_missing = copy_split_images(val_df, IMAGE_DIR, VAL_DIR)
test_copied, test_missing = copy_split_images(test_df, IMAGE_DIR, TEST_DIR)

print("Train copied:", train_copied, "missing:", train_missing)
print("Validation copied:", val_copied, "missing:", val_missing)
print("Test copied:", test_copied, "missing:", test_missing)

100%|██████████| 494/494 [05:13<00:00,  1.58it/s]

Train copied: 2304 missing: 0
Validation copied: 494 missing: 0
Test copied: 494 missing: 0


In [ ]:
train_df[["file_name", "Latitude", "Longitude"]].to_csv(
    os.path.join(TRAIN_DIR, "metadata.csv"),
    index=False
)

val_df[["file_name", "Latitude", "Longitude"]].to_csv(
    os.path.join(VAL_DIR, "metadata.csv"),
    index=False
)

test_df[["file_name", "Latitude", "Longitude"]].to_csv(
    os.path.join(TEST_DIR, "metadata.csv"),
    index=False
)

In [ ]:
from datasets import load_dataset

folder_path = "/content/drive/Shareddrives/CIS5190_final_project/processed_dataset_grouped"

dataset = load_dataset("imagefolder", data_dir=folder_path)
print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

Resolving data files:   0%|          | 0/2305 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/495 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/495 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'Latitude', 'Longitude'],
        num_rows: 2304
    })
    validation: Dataset({
        features: ['image', 'Latitude', 'Longitude'],
        num_rows: 494
    })
    test: Dataset({
        features: ['image', 'Latitude', 'Longitude'],
        num_rows: 494
    })
})
{'image': Image(mode=None, decode=True), 'Latitude': Value('float64'), 'Longitude': Value('float64')}
{'image': <PIL.Image.Image image mode=RGB size=4284x5712 at 0x7E6E74B42330>, 'Latitude': 39.95053611111111, 'Longitude': -75.19194166666666}
